# Import Libraries

In [14]:
import pandas as pd
import numpy as np
import spacy
import os
from tqdm import tqdm 
from sklearn.metrics import classification_report
import os
import requests
from openai import OpenAI, RateLimitError
import time
import random
from google import genai

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_colwidth', None)  # Show full content in each cell
pd.set_option('display.width', 1000)  # Set max width

# Load spaCy's English model
nlp = spacy.load('en_core_web_sm')

# Pre-Processing

In [15]:
label_mapper = {
    'knowledge' : 0,
    'comprehension' : 1,
    'application' : 2,
    'analysis' : 3,
    'synthesis' : 4,
    'evaluation' : 5
}

mapping = {
    'knowledge': 'knowledge',
    'remember': 'knowledge',
    'comprehension': 'comprehension',
    'understand': 'comprehension',
    'application': 'application',
    'apply': 'application',
    'analysis': 'analysis',
    'analyse': 'analysis',
    'evaluation': 'evaluation',
    'evaluate': 'evaluation',
    'synthesis': 'synthesis',
    'create': 'synthesis'
}

q_df = pd.read_csv(os.getcwd().replace('notebook' , 'dataset') + '/dataset4.csv')
queries = q_df['question']
q_df['label'] = q_df['label'].str.lower()
q_df['label'] = q_df['label'].replace(mapping)
label = q_df['label'].str.lower().map(label_mapper)
print(q_df['label'].value_counts())

label
synthesis        29
knowledge        22
evaluation       21
comprehension    20
analysis         19
application      15
Name: count, dtype: int64


# API Setup

In [16]:
# Sonar
 
api_key = os.environ.get("PERPLEXITY_API_KEY")

if api_key:
    print('successful')

url = "https://api.perplexity.ai/chat/completions"
headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

successful


In [4]:
# Groq

api_key = os.environ.get("GROQ_API_KEY")

if api_key:
    print('successful')

groq_client = OpenAI(
    base_url = "https://api.groq.com/openai/v1",
    api_key = api_key
)

successful


# Zero-Shot

## GPT-OSS-120B

In [8]:
zso_pred_labels = []

for query in tqdm(queries):
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""Given the query below, classify which Bloom's Taxonomy level it belongs to.
                    Levels: [knowledge, comprehension, application, analysis, synthesis, evaluation]
                
                query : {query}""",
            }
        ],
        model="openai/gpt-oss-120b",
    )

    reply = chat_completion.choices[0].message.content.lower()

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract the blooms level from previous reponse. Answer only in one word without punctuation from: 
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="openai/gpt-oss-20b",
        )

        reply = chat_completion.choices[0].message.content.lower()

    zso_pred_labels.append(reply.lower())

100%|██████████| 126/126 [04:48<00:00,  2.29s/it]


In [9]:
print(classification_report(label , [label_mapper[key.lower()] for key in zso_pred_labels]))

              precision    recall  f1-score   support

           0       0.88      0.95      0.91        22
           1       0.87      0.65      0.74        20
           2       0.45      0.33      0.38        15
           3       0.81      0.68      0.74        19
           4       0.65      0.90      0.75        29
           5       0.90      0.86      0.88        21

    accuracy                           0.76       126
   macro avg       0.76      0.73      0.74       126
weighted avg       0.77      0.76      0.75       126



## LLAMA4-Scout

In [10]:
zsl_pred_labels = []

for query in tqdm(queries):
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""Given the query below, classify which Bloom's Taxonomy level it belongs to.
                    Levels: [knowledge, comprehension, application, analysis, synthesis, evaluation]
                
                query : {query}""",
            }
        ],
        model="meta-llama/llama-4-scout-17b-16e-instruct",
    )

    reply = chat_completion.choices[0].message.content.lower()

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract the blooms level from previous reponse. Answer only in one word without punctuation from: 
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="meta-llama/llama-4-scout-17b-16e-instruct",
        )

        reply = chat_completion.choices[0].message.content.lower()

    zsl_pred_labels.append(reply.lower())

100%|██████████| 126/126 [09:21<00:00,  4.46s/it]


In [11]:
print(classification_report(label , [label_mapper[key.lower()] for key in zsl_pred_labels]))

              precision    recall  f1-score   support

           0       0.81      0.95      0.88        22
           1       0.75      0.60      0.67        20
           2       0.29      0.27      0.28        15
           3       0.80      0.63      0.71        19
           4       0.69      0.86      0.77        29
           5       0.84      0.76      0.80        21

    accuracy                           0.71       126
   macro avg       0.70      0.68      0.68       126
weighted avg       0.71      0.71      0.71       126



# FEW-Shot

## GPT-OSS-120B

In [ ]:
fso_pred_labels = []

for query in tqdm(queries):
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""Given the query below, classify which Bloom's Taxonomy level it belongs to.
                    Levels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Examples:

                    1. What is the capital of France? → Knowledge
                    2. Use Ohm’s law to calculate the current in a circuit. → Application
                    3. Critique the author’s argument in the article. → Evaluation
                    4. Summarize the main idea of the passage. → Comprehension             
                    5. Examine the causes of World War I. → Analysis                
                    6. Assess the validity of the research study’s conclusions. → Evaluation
                    7. Propose a plan to reduce plastic pollution in cities. → Synthesis
                    8. Define photosynthesis. → Knowledge
                    9. Apply the concept of supply and demand to predict price changes. → Application              
                    10. Create a new ending for the story. → Synthesis
                    11. Explain in your own words what Newton’s First Law means. → Comprehension
                    12. Identify the relationship between exercise and mental health in the study. → Analysis
                    13. Design an experiment to test the effect of sunlight on plant growth. → Synthesis
                    14. Differentiate between mitosis and meiosis. → Analysis
                    15. Solve the quadratic equation x² – 5x + 6 = 0. → Application
                    16. Judge whether the government’s policy on climate change is effective. → Evaluation
                    17. List the three states of matter. → Knowledge
                    18. Interpret the graph showing population growth. → Comprehension

                    Now classify the question into one Bloom’s Taxonomy level:

                    Question: {query}
                    """,
            }
        ],
        model="openai/gpt-oss-120b",
    )

    reply = chat_completion.choices[0].message.content.lower()

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract the blooms level from previous reponse. Answer only in one word without punctuation from: 
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="openai/gpt-oss-20b",
        )

        reply = chat_completion.choices[0].message.content.lower()

    fso_pred_labels.append(reply.lower())

In [13]:
print(classification_report(label , [label_mapper[key.lower()] for key in fso_pred_labels]))

              precision    recall  f1-score   support

           0       0.81      0.95      0.88        22
           1       0.82      0.70      0.76        20
           2       0.50      0.47      0.48        15
           3       0.80      0.63      0.71        19
           4       0.68      0.90      0.78        29
           5       1.00      0.76      0.86        21

    accuracy                           0.76       126
   macro avg       0.77      0.74      0.74       126
weighted avg       0.78      0.76      0.76       126



## LLAMA4-Scout

In [14]:
fsl_pred_labels = []

for query in tqdm(queries):
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""Classify question based on Bloom’s Taxonomy level using ONLY one word from:
                    [Knowledge, Comprehension, Application, Analysis, Synthesis, Evaluation]

                    Examples:

                    1. What is the capital of France? → Knowledge
                    2. Use Ohm’s law to calculate the current in a circuit. → Application
                    3. Critique the author’s argument in the article. → Evaluation
                    4. Summarize the main idea of the passage. → Comprehension             
                    5. Examine the causes of World War I. → Analysis                
                    6. Assess the validity of the research study’s conclusions. → Evaluation
                    7. Propose a plan to reduce plastic pollution in cities. → Synthesis
                    8. Define photosynthesis. → Knowledge
                    9. Apply the concept of supply and demand to predict price changes. → Application              
                    10. Create a new ending for the story. → Synthesis
                    11. Explain in your own words what Newton’s First Law means. → Comprehension
                    12. Identify the relationship between exercise and mental health in the study. → Analysis
                    13. Design an experiment to test the effect of sunlight on plant growth. → Synthesis
                    14. Differentiate between mitosis and meiosis. → Analysis
                    15. Solve the quadratic equation x² – 5x + 6 = 0. → Application
                    16. Judge whether the government’s policy on climate change is effective. → Evaluation
                    17. List the three states of matter. → Knowledge
                    18. Interpret the graph showing population growth. → Comprehension

                    Now classify the question into one Bloom’s Taxonomy level:

                    Question: {query}
                    """,
            }
        ],
        model="meta-llama/llama-4-scout-17b-16e-instruct",
    )

    reply = chat_completion.choices[0].message.content.lower()

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract the blooms level from previous reponse. Answer only in one word without punctuation from: 
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="meta-llama/llama-4-scout-17b-16e-instruct",
        )

        reply = chat_completion.choices[0].message.content.lower()

    fsl_pred_labels.append(reply.lower())

100%|██████████| 126/126 [03:50<00:00,  1.83s/it]


In [15]:
print(classification_report(label , [label_mapper[key.lower()] for key in fsl_pred_labels]))

              precision    recall  f1-score   support

           0       0.87      0.91      0.89        22
           1       0.79      0.75      0.77        20
           2       0.50      0.20      0.29        15
           3       0.87      0.68      0.76        19
           4       0.61      0.97      0.75        29
           5       0.88      0.71      0.79        21

    accuracy                           0.75       126
   macro avg       0.75      0.70      0.71       126
weighted avg       0.75      0.75      0.73       126



# Instruction Augmented

## GPT-OSS-120B

In [6]:
ipo_pred_labels = []

for query in tqdm(queries):
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""Classify question based on Bloom’s Taxonomy level using ONLY one word from:
                    Labels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Definations:
                    
                    1. Knowledge: Recalling facts, terms, basic concepts, or answers without necessarily understanding them
                    2. Comprehension: Demonstrating understanding of facts by interpreting, translating, summarizing, or explaining
                    3. Application: Using learned information in new concrete situations to solve problems
                    4. Analysis: Breaking down information into parts, examining relationships, distinguishing facts from inferences
                    5. Synthesis: Combining elements to form a new whole, proposing solutions, or designing new approaches
                    6. Evaluation: Making judgments based on criteria and standards through checking and critiquing

                    Now classify the question into one Bloom’s Taxonomy level:

                    Question: {query}
                    """,
            }
        ],
        model="openai/gpt-oss-120b",
    )

    reply = chat_completion.choices[0].message.content.lower()

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract the blooms level from previous reponse. Answer only in one word without punctuation from: 
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="openai/gpt-oss-20b",
        )

        reply = chat_completion.choices[0].message.content.lower()

    ipo_pred_labels.append(reply.lower())

100%|██████████| 126/126 [05:00<00:00,  2.38s/it]


In [7]:
print(classification_report(label , [label_mapper[key.lower()] for key in ipo_pred_labels]))

              precision    recall  f1-score   support

           0       0.81      0.95      0.88        22
           1       0.68      0.75      0.71        20
           2       0.54      0.47      0.50        15
           3       0.86      0.63      0.73        19
           4       0.76      0.86      0.81        29
           5       0.94      0.81      0.87        21

    accuracy                           0.77       126
   macro avg       0.76      0.75      0.75       126
weighted avg       0.77      0.77      0.77       126



## LLAMA4-Scout

In [8]:
ipl_pred_labels = []

for query in tqdm(queries):
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""Classify question based on Bloom’s Taxonomy level using ONLY one word from:
                    Labels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Definations:
                    
                    1. Knowledge: Recalling facts, terms, basic concepts, or answers without necessarily understanding them
                    2. Comprehension: Demonstrating understanding of facts by interpreting, translating, summarizing, or explaining
                    3. Application: Using learned information in new concrete situations to solve problems
                    4. Analysis: Breaking down information into parts, examining relationships, distinguishing facts from inferences
                    5. Synthesis: Combining elements to form a new whole, proposing solutions, or designing new approaches
                    6. Evaluation: Making judgments based on criteria and standards through checking and critiquing

                    Now classify the question into one Bloom’s Taxonomy level:

                    Question: {query}
                    """,
            }
        ],
        model="meta-llama/llama-4-scout-17b-16e-instruct",
    )

    reply = chat_completion.choices[0].message.content.lower()

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract the blooms level from previous reponse. Answer only in one word without punctuation from: 
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="meta-llama/llama-4-scout-17b-16e-instruct",
        )

        reply = chat_completion.choices[0].message.content.lower()

    ipl_pred_labels.append(reply.lower())

100%|██████████| 126/126 [05:02<00:00,  2.40s/it]


In [9]:
print(classification_report(label , [label_mapper[key.lower()] for key in ipl_pred_labels]))

              precision    recall  f1-score   support

           0       0.88      0.95      0.91        22
           1       0.83      0.75      0.79        20
           2       0.36      0.27      0.31        15
           3       0.88      0.74      0.80        19
           4       0.68      0.90      0.78        29
           5       0.84      0.76      0.80        21

    accuracy                           0.76       126
   macro avg       0.75      0.73      0.73       126
weighted avg       0.76      0.76      0.75       126



# Defination-Augmented-Few-Shot

## GPT-OSS-120B

In [17]:
ifso_pred_labels = []

for query in tqdm(queries):
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""Classify question based on Bloom’s Taxonomy level using ONLY one word from:
                    Labels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Definations:
                    
                    1. Knowledge: Recalling facts, terms, basic concepts, or answers without necessarily understanding them
                    2. Comprehension: Demonstrating understanding of facts by interpreting, translating, summarizing, or explaining
                    3. Application: Using learned information in new concrete situations to solve problems
                    4. Analysis: Breaking down information into parts, examining relationships, distinguishing facts from inferences
                    5. Synthesis: Combining elements to form a new whole, proposing solutions, or designing new approaches
                    6. Evaluation: Making judgments based on criteria and standards through checking and critiquing

                    Examples:

                    1. What is the capital of France? → Knowledge
                    2. Use Ohm’s law to calculate the current in a circuit. → Application
                    3. Critique the author’s argument in the article. → Evaluation
                    4. Summarize the main idea of the passage. → Comprehension             
                    5. Examine the causes of World War I. → Analysis                
                    6. Assess the validity of the research study’s conclusions. → Evaluation
                    7. Propose a plan to reduce plastic pollution in cities. → Synthesis
                    8. Define photosynthesis. → Knowledge
                    9. Apply the concept of supply and demand to predict price changes. → Application              
                    10. Create a new ending for the story. → Synthesis
                    11. Explain in your own words what Newton’s First Law means. → Comprehension
                    12. Identify the relationship between exercise and mental health in the study. → Analysis
                    13. Design an experiment to test the effect of sunlight on plant growth. → Synthesis
                    14. Differentiate between mitosis and meiosis. → Analysis
                    15. Solve the quadratic equation x² – 5x + 6 = 0. → Application
                    16. Judge whether the government’s policy on climate change is effective. → Evaluation
                    17. List the three states of matter. → Knowledge
                    18. Interpret the graph showing population growth. → Comprehension
                    Now classify the question into one Bloom’s Taxonomy level:

                    Question: {query}
                    """,
            }
        ],
        model="openai/gpt-oss-120b",
    )

    reply = chat_completion.choices[0].message.content.lower()

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract the blooms level from previous reponse. Answer only in one word without punctuation from: 
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="openai/gpt-oss-20b",
        )

        reply = chat_completion.choices[0].message.content.lower()

    ifso_pred_labels.append(reply.lower())

100%|██████████| 126/126 [09:51<00:00,  4.69s/it]


In [18]:
print(classification_report(label , [label_mapper[key.lower()] for key in ifso_pred_labels]))

              precision    recall  f1-score   support

           0       0.83      0.91      0.87        22
           1       0.70      0.70      0.70        20
           2       0.42      0.33      0.37        15
           3       0.86      0.63      0.73        19
           4       0.68      0.90      0.78        29
           5       0.94      0.81      0.87        21

    accuracy                           0.75       126
   macro avg       0.74      0.71      0.72       126
weighted avg       0.75      0.75      0.74       126



## LLAMA4-Scout

In [19]:
ifsl_pred_labels = []

for query in tqdm(queries):
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""Classify question based on Bloom’s Taxonomy level using ONLY one word from:
                    Labels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Definations:
                    
                    1. Knowledge: Recalling facts, terms, basic concepts, or answers without necessarily understanding them
                    2. Comprehension: Demonstrating understanding of facts by interpreting, translating, summarizing, or explaining
                    3. Application: Using learned information in new concrete situations to solve problems
                    4. Analysis: Breaking down information into parts, examining relationships, distinguishing facts from inferences
                    5. Synthesis: Combining elements to form a new whole, proposing solutions, or designing new approaches
                    6. Evaluation: Making judgments based on criteria and standards through checking and critiquing

                    Examples:

                    1. What is the capital of France? → Knowledge
                    2. Use Ohm’s law to calculate the current in a circuit. → Application
                    3. Critique the author’s argument in the article. → Evaluation
                    4. Summarize the main idea of the passage. → Comprehension             
                    5. Examine the causes of World War I. → Analysis                
                    6. Assess the validity of the research study’s conclusions. → Evaluation
                    7. Propose a plan to reduce plastic pollution in cities. → Synthesis
                    8. Define photosynthesis. → Knowledge
                    9. Apply the concept of supply and demand to predict price changes. → Application              
                    10. Create a new ending for the story. → Synthesis
                    11. Explain in your own words what Newton’s First Law means. → Comprehension
                    12. Identify the relationship between exercise and mental health in the study. → Analysis
                    13. Design an experiment to test the effect of sunlight on plant growth. → Synthesis
                    14. Differentiate between mitosis and meiosis. → Analysis
                    15. Solve the quadratic equation x² – 5x + 6 = 0. → Application
                    16. Judge whether the government’s policy on climate change is effective. → Evaluation
                    17. List the three states of matter. → Knowledge
                    18. Interpret the graph showing population growth. → Comprehension
                    Now classify the question into one Bloom’s Taxonomy level:

                    Question: {query}
                    """,
            }
        ],
        model="meta-llama/llama-4-scout-17b-16e-instruct",
    )

    reply = chat_completion.choices[0].message.content.lower()

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract the blooms level from previous reponse. Answer only in one word without punctuation from: 
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="meta-llama/llama-4-scout-17b-16e-instruct",
        )

        reply = chat_completion.choices[0].message.content.lower()

    ifsl_pred_labels.append(reply.lower())

100%|██████████| 126/126 [04:55<00:00,  2.35s/it]


In [20]:
print(classification_report(label , [label_mapper[key.lower()] for key in ifsl_pred_labels]))

              precision    recall  f1-score   support

           0       0.83      0.91      0.87        22
           1       0.74      0.70      0.72        20
           2       0.40      0.13      0.20        15
           3       0.82      0.74      0.78        19
           4       0.64      0.97      0.77        29
           5       0.88      0.71      0.79        21

    accuracy                           0.74       126
   macro avg       0.72      0.69      0.69       126
weighted avg       0.73      0.74      0.72       126



# Chain-of-Thought

## GPT-OSS-120B

In [21]:
coto_pred_labels = []

for query in tqdm(queries):
    # Reason

    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""
                    Given the query below, reason about which Bloom's Taxonomy level it belongs to.
                    Levels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Query: {query}""",
            }
        ],
        model="openai/gpt-oss-120b",
    )

    reply = chat_completion.choices[0].message.content.lower()

    # Summarize and Classify
    
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""
                    Summarize the reasoning into exactly one Bloom's Taxonomy level.
                    Levels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Reasoning: {reply}

                    Answer in one word ONLY""",
            }
        ],
        model="openai/gpt-oss-120b",
    )

    reply = chat_completion.choices[0].message.content.lower()

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract the blooms level from previous reponse. Answer only in one word without punctuation from: 
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="openai/gpt-oss-20b",
        )

        reply = chat_completion.choices[0].message.content.lower()
    time.sleep(5 * random.randint(0, 2))

    coto_pred_labels.append(reply.lower())

100%|██████████| 126/126 [12:00<00:00,  5.72s/it]


In [22]:
print(classification_report(label , [label_mapper[key.lower()] for key in coto_pred_labels]))

              precision    recall  f1-score   support

           0       0.87      0.91      0.89        22
           1       1.00      0.65      0.79        20
           2       0.45      0.33      0.38        15
           3       0.72      0.68      0.70        19
           4       0.63      0.90      0.74        29
           5       0.90      0.86      0.88        21

    accuracy                           0.75       126
   macro avg       0.76      0.72      0.73       126
weighted avg       0.77      0.75      0.75       126



## LLAMA4-Scout

In [10]:
cotl_pred_labels = []

for query in tqdm(queries):
    # Reason

    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""
                    Given the query below, reason about which Bloom's Taxonomy level it belongs to.
                    Levels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Query: {query}""",
            }
        ],
        model="meta-llama/llama-4-scout-17b-16e-instruct",
    )

    reply = chat_completion.choices[0].message.content.lower()

    # Summarize and Classify
    
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""
                    Summarize the reasoning into exactly one Bloom's Taxonomy level.
                    Levels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Reasoning: {reply}

                    Answer in one word ONLY""",
            }
        ],
        model="meta-llama/llama-4-scout-17b-16e-instruct",
    )

    reply = chat_completion.choices[0].message.content.lower()

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract the blooms level from previous reponse. Answer only in one word without punctuation from: 
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="meta-llama/llama-4-scout-17b-16e-instruct",
        )

        reply = chat_completion.choices[0].message.content.lower()
    time.sleep(5 * random.randint(0, 2))

    cotl_pred_labels.append(reply.lower())

100%|██████████| 126/126 [13:51<00:00,  6.60s/it]


In [11]:
print(classification_report(label , [label_mapper[key.lower()] for key in cotl_pred_labels]))

              precision    recall  f1-score   support

           0       0.90      0.86      0.88        22
           1       0.82      0.70      0.76        20
           2       0.46      0.40      0.43        15
           3       0.78      0.74      0.76        19
           4       0.65      0.90      0.75        29
           5       0.94      0.76      0.84        21

    accuracy                           0.75       126
   macro avg       0.76      0.73      0.74       126
weighted avg       0.77      0.75      0.75       126



# Save Labels

In [ ]:
label_data = {
    'zero_shot_context_gpt' : zso_pred_labels,
    'zero_shot_context_llama' : zsl_pred_labels,
    'few_shot_context_gpt' : fso_pred_labels, 
    'few_shot_context_llama' : fsl_pred_labels,
    'cot_context_gpt' : coto_pred_labels,
    'cot_context_llama' : cotl_pred_labels,
    'dazs_context_gpt' : ipo_pred_labels,
    'dazs_context_llama' : ipl_pred_labels,
    'defination_few_shot_context_gpt' : ifso_pred_labels,
    'defination_few_shot_context_llama' : ifsl_pred_labels
            }

df = pd.DataFrame(data= label_data)

df.to_csv('context_groq.csv', index=False) 

In [21]:
label_data = {
    'defination_few_shot_context_gpt' : ifso_pred_labels,
    'defination_few_shot_context_llama' : ifsl_pred_labels
            }

df = pd.DataFrame(data= label_data)

label_df = pd.read_csv('context.csv')
df = pd.concat([label_df, df], axis=1)

df.to_csv('context.csv', index=False) 